# Adding Functions

Functions are Python scripts that run on the DataLab server against your files. You write them locally, wrap them with their dependencies, and upload them. Once uploaded, a function can be used as a processing step inside a pipeline. This tutorial shows you how to define, test locally, and upload a function.

## Setup

In [ ]:
from pathlib import Path

import gfhub
import pandas as pd

client = gfhub.Client()

## The function signature

DataLab functions follow a specific convention. Input files are positional-only parameters (before the `/`), typed as `Path`. Configuration parameters are keyword-only (after the `*`), with default values. When the function produces a single output file, it returns a `Path` directly. When it produces multiple outputs, it returns a `dict[str, Path]`.

This convention tells DataLab which arguments are file inputs and which are configuration, so it can wire files through a pipeline automatically.

In [ ]:
def csv_to_parquet_example(input_file: Path, /) -> Path:
    """Convert a CSV measurement file to Parquet format."""
    df = pd.read_csv(input_file)
    output = input_file.with_suffix(".parquet")
    df.to_parquet(output, index=False)
    return output

## Wrapping with dependencies

Before uploading, you wrap the function in a `gfhub.Function` and declare its third-party dependencies. This is similar to a `requirements.txt` scoped to just this function. The keys are package specs (with optional version constraints), and the values are the import statements that make those packages available inside the function body.

If you forget to declare a dependency that your function uses, `gfhub.Function` will raise an error immediately, before anything is uploaded.

In [ ]:
func = gfhub.Function(
    csv_to_parquet_example,
    dependencies={"pandas[pyarrow]": "import pandas as pd"},
)

print(func)

## Test locally before uploading

You can run the function on a local file before uploading it to the server using the method `.eval()`. This uses `uv run` under the hood to install dependencies in an isolated environment and execute the function exactly as DataLab would. It is a good habit to verify your function works correctly before uploading it.

In [ ]:
# Create a small test file
test_csv = Path("test_measurements.csv")
test_csv.write_text("wavelength_nm,insertion_loss_db\n1550,0.4\n1560,0.5\n1570,0.45\n")

result = func.eval(test_csv)
print(result)
# {"success": True, "output": "/path/to/test_measurements.parquet"}

# Clean up test files
test_csv.unlink(missing_ok=True)
Path(result["output"]).unlink(missing_ok=True)

## Upload the function

Once you are happy with the function, upload it. If a function with the same name already exists it will be updated by default. The returned object contains the function `id` and metadata.

In [ ]:
import json

result = client.add_function(func)
print(json.dumps(result, indent=2, default=str))

## List uploaded functions

You can list all functions available in your organization. This is useful to confirm an upload succeeded or to find function names when referencing them inside a pipeline.

In [ ]:
functions = client.list_functions()

print(f"Available functions ({len(functions)}):")
for f in functions:
    print(f"  - {f['name']}  (id={f['id']})")